# Next step: step 9, failures and status
**What it adds:** a bad simulation no longer stops the run, and every simulation reports what happened.

- A simulation (one model on one simulator entry) **fails as a whole** when computing any requested item raises an
  error, or when a simulated array holds NaN or inf. `sim.timing` isn't checked: it legitimately holds NaN.
- A failure: its cells hold `None`, nothing is saved for it, nothing goes in `model.json` (so a later run retries it),
  and the error is logged with its traceback. The run moves on.
- **Errors while writing files stop the run:** a full disk would only fail everything after it.
- Per stage (`"sim"` so far), indexed `[m, s]` like the data:

| Field | Each cell |
|---|---|
| `result.status["sim"]` | `"ran"`, `"reused"` (saved, and nothing had to be computed) or `"failed"` |
| `result.seconds["sim"]` | Wall time, loading included |
| `result.errors["sim"]` | `"Type: message"` for a failure, else `None` |

- A run that saves adds them to its record in `index.json` when it ends (only then: simple, and revisited in Step 12).
- The run ends with a log line counting what ran, was reused and failed. Later stages will mark work that depends on a
  failed simulation `"skipped"`.

## What changed
**`src/galactic_tidal_constraints/simr/simr.py`.** Each simulation's loading and computing moved into `_simulate`,
which also checks the arrays. The loop catches its errors, records them and moves on; saving stays outside the `try`,
so write errors still stop the run:
```python
start = time.perf_counter()
...
try:
    values, computed = _simulate(params, config, returned, to_save, stored or {})
except Exception as error:      # a failed simulation: recorded, and the run moves on
    logger.exception("model %d (id %s) on simulator %d failed", m, ids[m], s)
    status["sim"][m, s] = "failed"
    errors["sim"][m, s] = f"{type(error).__name__}: {error}"
    seconds["sim"][m, s] = time.perf_counter() - start
    continue
for item in returned:
    data[item][m, s] = values[item]
if to_save:                     # errors while writing aren't caught: they stop the run
    _save_simulation(...)
status["sim"][m, s] = "reused" if stored and not computed else "ran"
seconds["sim"][m, s] = time.perf_counter() - start
```
```python
def _simulate(params, config, returned, to_save, stored):
    values = {item: stored[item]() for item in returned if item in stored}    # load what's saved
    computed = False
    for item in dict.fromkeys(returned + to_save):
        if item not in values:
            values[item] = SIM_DATA[item](params, config)                     # compute the rest
            computed = True
    for item, value in values.items():
        if isinstance(value, np.ndarray) and not np.isfinite(value).all():
            raise ValueError(f"NaN or inf in {item}")
    return values, computed
```
`SimrResult` gains `status`, `seconds` and `errors`; the run record in `index.json` gains the same, as lists.

**Tests:** `tests/test_simr.py`, section "Step 9". **Docs:** `docs/design/simr.md`, Failures and status.
**Decisions:** `PLAN.md`, Structure (Failures, Status).

In [ ]:
%matplotlib inline
import logging
import tempfile
from pathlib import Path

import numpy as np

from galactic_tidal_constraints.simr import simr
from galactic_tidal_constraints.simr import simulation as sim

# The notebook is the application here, so it sets up logging (simr itself never does).
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

# Line 3 of JSPAM's archive/data.txt; the same with mass2 doubled; and one approaching now and unbound, which has
# no pass before now, so with no_pass="error" it fails.
params = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                   12.97415, 135.11902, 97.54035, 305.01225])
heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2
first_approach = params.copy()
first_approach[3:6] *= -3
models = np.stack([params, first_approach, heavier])
simulators = [{"n1": 500, "n2": 500, "no_pass": "error"}, {"n1": 500, "n2": 500}]   # entry 1 falls back to -5

## 1. A failure doesn't stop the run
Model 1 fails on entry 0 (no pass, and `no_pass="error"`); on entry 1 it falls back to a start at −5 and runs. The
traceback and the count are in the log above the output:

In [ ]:
result = simr.run(models, {"run": {"return": ["sim.orbit"]}, "simulator": simulators})
print("status:\n", result.status["sim"])
print("orbit cells:", [[None if cell is None else cell.shape for cell in row] for row in result.data["sim.orbit"]])
print("errors[1, 0]:", result.errors["sim"][1, 0])
print("seconds:\n", result.seconds["sim"].round(3))
print("failed:", (result.status["sim"] == "failed").sum())

## 2. Saving: failures aren't saved, and are retried

In [ ]:
output = Path(tempfile.mkdtemp(prefix="simr_9_"))
first = simr.run(models, {"run": {"save": ["sim.particles"]}, "simulator": simulators}, output=output)
record = simr.load_model(output / f"model_{first.ids[1]}")
print("model 1's saved simulations:", list(record["simulations"]), "(sim_00 failed, so it isn't there)")

[run_record] = simr.load_index(output)["runs"]
print("index status:", run_record["status"])
print("index errors:", [[e and e[:50] for e in row] for row in run_record["errors"]["sim"]])

The same request again: everything saved is reused, and model 1's entry 0 is tried again (and fails again):

In [ ]:
again = simr.run(models, {"run": {"save": ["sim.particles"]}, "simulator": simulators}, output=output)
print("status:\n", again.status["sim"])
print("seconds:\n", again.seconds["sim"].round(3))

## 3. NaN or inf counts as a failure
Real SPAM runs rarely produce NaN, so this swaps the orbit for one with a NaN in it, just to show the check:

In [ ]:
real_orbit = simr.SIM_DATA["sim.orbit"]
simr.SIM_DATA["sim.orbit"] = lambda params, config: np.full((3, 7), np.nan, dtype=np.float32)
try:
    broken = simr.run(models[:1], {"run": {"return": ["sim.orbit", "sim.timing"]}, "simulator": simulators[1]})
finally:
    simr.SIM_DATA["sim.orbit"] = real_orbit
print("status:", broken.status["sim"][0, 0], "  error:", broken.errors["sim"][0, 0])
print("timing cell too:", broken.data["sim.timing"][0, 0], "(the whole simulation fails)")